## Q-1


In [240]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer
from scipy.stats import zscore, mstats
import sqlite3
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler

In [241]:
riders_df = pd.read_csv('riders.csv')
trips_df = pd.read_json('trips.json')

conn = sqlite3.connect('city_zones.db')
city_df = pd.read_sql_query("SELECT * FROM city_zones", conn)
conn.close()

### Display 

In [242]:
## First 5 rows of each dataset
datasets = [riders_df, trips_df, city_df]
for df in datasets:
    print('\n',df.head())


   rider_id  age  gender signup_date customer_ride_frequency
0    R_001   19   Other  2023-09-30                     Low
1    R_002   33     NaN  2023-03-06                    High
2    R_003   65     NaN  2023-03-07                     Low
3    R_004   65   Other  2023-12-04                    High
4    R_005   23  Female  2023-09-01                    High

   trip_id rider_id zone_id      pickup_datetime  distance  duration  \
0  T_0000    R_004    Z_03  2024-04-12 02:11:00      4.99     21.72   
1  T_0001    R_032    Z_03  2024-04-20 14:02:00     17.72     74.55   
2  T_0002    R_011    Z_01  2024-04-15 14:32:00      2.37      8.97   
3  T_0003    R_003    Z_10  2024-04-12 00:54:00     10.69     31.53   
4  T_0004    R_021    Z_02           04/12/2024      6.73     15.48   

   fare_amount  
0       146.83  
1       694.98  
2        79.08  
3       509.53  
4       173.85  

   zone_id     zone_name traffic_level  population_density
0    Z_01      Downtown          High          

In [243]:
## Info
for df in datasets:
    print('\n',df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   rider_id                 50 non-null     object
 1   age                      50 non-null     int64 
 2   gender                   39 non-null     object
 3   signup_date              50 non-null     object
 4   customer_ride_frequency  50 non-null     object
dtypes: int64(1), object(4)
memory usage: 2.1+ KB

 None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   trip_id          200 non-null    object 
 1   rider_id         200 non-null    object 
 2   zone_id          200 non-null    object 
 3   pickup_datetime  200 non-null    object 
 4   distance         192 non-null    float64
 5   duration         192 non-null    float6

In [244]:
## Isnull values
for df in datasets:
    print('\n',df.isnull().sum())


 rider_id                    0
age                         0
gender                     11
signup_date                 0
customer_ride_frequency     0
dtype: int64

 trip_id            0
rider_id           0
zone_id            0
pickup_datetime    0
distance           8
duration           8
fare_amount        7
dtype: int64

 zone_id               0
zone_name             0
traffic_level         0
population_density    0
dtype: int64


### Duplicates 

In [245]:
for df in datasets:
    print('\n', df.duplicated().sum())


 0

 0

 0


In [246]:
## Invalid input

if 'age' in riders_df.columns:
    invalid_ages = riders_df[(riders_df['age'] < 10)]
    print("\nInvalid ages:\n", invalid_ages)


Invalid ages:
   rider_id  age gender signup_date customer_ride_frequency
6    R_007   -2    NaN  2023-03-26                     Med
8    R_009   -2   Male  2023-09-28                     Med


In [247]:
if 'distance' in trips_df.columns:
    invalid_distances = trips_df[(trips_df['distance'] < 0)]
    print("\nInvalid distances:\n", invalid_distances)


Invalid distances:
 Empty DataFrame
Columns: [trip_id, rider_id, zone_id, pickup_datetime, distance, duration, fare_amount]
Index: []


## Q-2 Data Cleaning

In [248]:
## Missing value using SimpleImputer mean
imputer = SimpleImputer(strategy='mean')
trips_df['fare_amount'] = imputer.fit_transform(trips_df[['fare_amount']])

print("\nMissing values after mean imputation:\n", trips_df['fare_amount'].isnull().sum())



Missing values after mean imputation:
 0


In [249]:
## Most frequent strategy for categorical missing data
imputer_freq = SimpleImputer(strategy='most_frequent')
riders_df[['gender']] = imputer_freq.fit_transform(riders_df[['gender']])

print("\nMissing values after most frequent imputation:\n", riders_df['gender'].isnull().sum())


Missing values after most frequent imputation:
 0


In [250]:
## KNN Imputer 

knn_imputer = KNNImputer(n_neighbors=5)
cols = ['duration', 'distance', 'fare_amount']
trips_df[cols] = knn_imputer.fit_transform(trips_df[cols])

print("\nMissing values after KNN imputation:\n", trips_df[cols].isnull().sum())


Missing values after KNN imputation:
 duration       0
distance       0
fare_amount    0
dtype: int64


In [251]:
## convert inconsistent data types
trips_df['pickup_datetime'] = pd.to_datetime(trips_df['pickup_datetime'], errors='coerce')

print("\nData types after conversion:\n", trips_df.dtypes)


Data types after conversion:
 trip_id                    object
rider_id                   object
zone_id                    object
pickup_datetime    datetime64[ns]
distance                  float64
duration                  float64
fare_amount               float64
dtype: object


In [252]:
## remove unrealistic entries 

riders_df = riders_df[riders_df['age'] >= 10]
trips_df = trips_df[trips_df['fare_amount'] >= 0 & (trips_df['distance'] >= 0) & (trips_df['duration'] >= 0)]

print("\nData after removing unrealistic entries:\n")
print("Riders:\n", riders_df.describe())


Data after removing unrealistic entries:

Riders:
              age
count  48.000000
mean   42.479167
std    14.896793
min    18.000000
25%    31.750000
50%    40.500000
75%    58.000000
max    65.000000


## Q-3 Outlier Handling

In [253]:
## Z-score for Fare & Distance

z_scores = zscore(trips_df[['fare_amount', 'distance']])
filtered_entries = (np.abs(z_scores) < 3).all(axis=1)
trips_df = trips_df[filtered_entries]

print("\nData after Z-score outlier removal:\n")
print(trips_df.describe())


Data after Z-score outlier removal:

                     pickup_datetime    distance    duration  fare_amount
count                            185  195.000000  195.000000   195.000000
mean   2024-04-16 23:04:57.081081344   11.883549   41.486697   362.582673
min              2024-04-01 11:47:00    0.000000    0.000000    25.460000
25%              2024-04-09 07:21:00    6.750000   20.420000   149.625000
50%              2024-04-16 17:58:00   11.370000   35.700000   321.350000
75%              2024-04-24 10:35:00   17.380000   59.910000   521.130000
max              2024-05-01 05:57:00   24.910000  117.220000  1050.830000
std                              NaN    6.988160   26.892106   247.944957


In [254]:
## IQR for Ride Duration

Q1 = trips_df['duration'].quantile(0.25)
Q3 = trips_df['duration'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
trips_df = trips_df[(trips_df['duration'] >= lower_bound) & (trips_df['duration'] <= upper_bound)]

print("\nData after IQR outlier removal:\n")
print(trips_df.describe())


Data after IQR outlier removal:

                     pickup_datetime    distance    duration  fare_amount
count                            185  195.000000  195.000000   195.000000
mean   2024-04-16 23:04:57.081081344   11.883549   41.486697   362.582673
min              2024-04-01 11:47:00    0.000000    0.000000    25.460000
25%              2024-04-09 07:21:00    6.750000   20.420000   149.625000
50%              2024-04-16 17:58:00   11.370000   35.700000   321.350000
75%              2024-04-24 10:35:00   17.380000   59.910000   521.130000
max              2024-05-01 05:57:00   24.910000  117.220000  1050.830000
std                              NaN    6.988160   26.892106   247.944957


In [255]:
## Winsorization for Extreme surge fares

trips_df['fare_amount'] = mstats.winsorize(trips_df['fare_amount'], limits=[0.05, 0.05])
print("\nData after Winsorization:\n")
print(trips_df.describe())


Data after Winsorization:

                     pickup_datetime    distance    duration  fare_amount
count                            185  195.000000  195.000000   195.000000
mean   2024-04-16 23:04:57.081081344   11.883549   41.486697   357.970725
min              2024-04-01 11:47:00    0.000000    0.000000    45.910000
25%              2024-04-09 07:21:00    6.750000   20.420000   149.625000
50%              2024-04-16 17:58:00   11.370000   35.700000   321.350000
75%              2024-04-24 10:35:00   17.380000   59.910000   521.130000
max              2024-05-01 05:57:00   24.910000  117.220000   868.080000
std                              NaN    6.988160   26.892106   235.385200


C:\Users\victus\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\_function_base_impl.py:4859: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
C:\Users\victus\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\_function_base_impl.py:4859: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
C:\Users\victus\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\_function_base_impl.py:4859: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(


## Q-4 Data Transformation

In [256]:
## Convert DateTime

trips_df['hour'] = trips_df['pickup_datetime'].dt.hour
trips_df['day_of_week'] = trips_df['pickup_datetime'].dt.day_name()
trips_df['month'] = trips_df['pickup_datetime'].dt.month_name()

print("\nData after DateTime feature extraction:\n")
print(trips_df.head())


Data after DateTime feature extraction:

  trip_id rider_id zone_id     pickup_datetime  distance  duration  \
0  T_0000    R_004    Z_03 2024-04-12 02:11:00      4.99     21.72   
1  T_0001    R_032    Z_03 2024-04-20 14:02:00     17.72     74.55   
2  T_0002    R_011    Z_01 2024-04-15 14:32:00      2.37      8.97   
3  T_0003    R_003    Z_10 2024-04-12 00:54:00     10.69     31.53   
4  T_0004    R_021    Z_02                 NaT      6.73     15.48   

   fare_amount  hour day_of_week  month  
0       146.83   2.0      Friday  April  
1       694.98  14.0    Saturday  April  
2        79.08  14.0      Monday  April  
3       509.53   0.0      Friday  April  
4       173.85   NaN         NaN    NaN  


In [257]:
## Encode categorical columns

## Label Encoding
label_encoder = LabelEncoder()
riders_df['gender'] = label_encoder.fit_transform(riders_df['gender'])
print("\nData after Label Encoding:\n")
print(riders_df.head())

## One-Hot Encoding for ride_payment_mode, zone_name
zone_df = pd.get_dummies(city_df['zone_name'], prefix='zone_name')
city_df = pd.concat([city_df, zone_df], axis=1)
print("\nData after One-Hot Encoding:\n")
print(city_df.head())

## Ordinal Encoding for traffic_level 
traffic_mapping = {'Low': 1, 'Medium': 2, 'High': 3}
city_df['traffic_level'] = city_df['traffic_level'].map(traffic_mapping)
print("\nData after Ordinal Encoding:\n")
print(city_df.head())


Data after Label Encoding:

  rider_id  age  gender signup_date customer_ride_frequency
0    R_001   19       2  2023-09-30                     Low
1    R_002   33       1  2023-03-06                    High
2    R_003   65       1  2023-03-07                     Low
3    R_004   65       2  2023-12-04                    High
4    R_005   23       0  2023-09-01                    High

Data after One-Hot Encoding:

  zone_id     zone_name traffic_level  population_density  zone_name_Airport  \
0    Z_01      Downtown          High                 752              False   
1    Z_02    North Park           Low                2892              False   
2    Z_03  East Village           Low                2751              False   
3    Z_04       Suburbs           Low                4904              False   
4    Z_05       Airport        Medium                1877               True   

   zone_name_Downtown  zone_name_East Village  zone_name_Harbor  \
0                True           

C:\Users\victus\AppData\Local\Temp\ipykernel_614828\2772284334.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  riders_df['gender'] = label_encoder.fit_transform(riders_df['gender'])


In [258]:
## Binnig customer ride frequency into categories
import numpy as np

# 1. Calculate the frequency (count trips per rider)
ride_counts = trips_df.groupby('rider_id').size().reset_index(name='total_trips')

# 2. Merge this count back into your riders dataframe
df_riders = riders_df.merge(ride_counts, on='rider_id', how='left')

# 3. Fill NaN with 0 (for riders who signed up but haven't booked a ride yet)
df_riders['total_trips'] = df_riders['total_trips'].fillna(0)

# 4. Define Bins and Labels
# Logic: 0-5 rides = Low, 6-15 rides = Med, 16+ = High
bins = [-1, 5, 15, np.inf] 
labels = ['Low', 'Med', 'High']

# 5. Apply pd.cut to create the 'customer_ride_frequency' feature
df_riders['customer_ride_frequency'] = pd.cut(
    df_riders['total_trips'], 
    bins=bins, 
    labels=labels
)

# Display result
print(df_riders[['rider_id', 'total_trips', 'customer_ride_frequency']].head())

  rider_id  total_trips customer_ride_frequency
0    R_001          1.0                     Low
1    R_002          4.0                     Low
2    R_003          4.0                     Low
3    R_004          6.0                     Med
4    R_005          1.0                     Low


In [259]:
## transform skewed numeric columns 

### Log transformation for fare_amount
trips_df['fare_amount_log'] = np.log1p(trips_df['fare_amount'])
print("\nData after log transformation:")
print(trips_df[['fare_amount', 'fare_amount_log']].head())

### Squareroot transformation on duration
trips_df['duration_sqrt'] = np.sqrt(trips_df['duration'])
print("\nData after square root transformation:")
print(trips_df[['duration', 'duration_sqrt']].head())



Data after log transformation:
   fare_amount  fare_amount_log
0       146.83         4.996063
1       694.98         6.545321
2        79.08         4.383026
3       509.53         6.235449
4       173.85         5.163928

Data after square root transformation:
   duration  duration_sqrt
0     21.72       4.660472
1     74.55       8.634234
2      8.97       2.994996
3     31.53       5.615158
4     15.48       3.934463


## Q-5 Feature Scaling

In [260]:
## Scale numeric feature using Standard Scaler
cols_to_scale = ['fare_amount', 'distance', 'duration']
print("\nData before scaling:")
print(trips_df[cols_to_scale].head())

print("\nBefore scaling:")
print(trips_df[cols_to_scale].describe())


scaler = StandardScaler()
trips_df[cols_to_scale] = scaler.fit_transform(trips_df[cols_to_scale])
print("\nData after scaling:")
print(trips_df[cols_to_scale].head())

print("\nAfter scaling:")
print(trips_df[cols_to_scale].describe())


Data before scaling:
   fare_amount  distance  duration
0       146.83      4.99     21.72
1       694.98     17.72     74.55
2        79.08      2.37      8.97
3       509.53     10.69     31.53
4       173.85      6.73     15.48

Before scaling:
       fare_amount    distance    duration
count   195.000000  195.000000  195.000000
mean    357.970725   11.883549   41.486697
std     235.385200    6.988160   26.892106
min      45.910000    0.000000    0.000000
25%     149.625000    6.750000   20.420000
50%     321.350000   11.370000   35.700000
75%     521.130000   17.380000   59.910000
max     868.080000   24.910000  117.220000

Data after scaling:
   fare_amount  distance  duration
0    -0.899310 -0.989000 -0.736929
1     1.435420  0.837341  1.232645
2    -1.187877 -1.364885 -1.212266
3     0.645535 -0.171235 -0.371199
4    -0.784224 -0.739367 -0.969565

After scaling:
        fare_amount      distance      duration
count  1.950000e+02  1.950000e+02  1.950000e+02
mean  -1.275333e-16  

In [261]:
## MinMAx Scaling for age

min_max_scaler = MinMaxScaler()
riders_df['age_minmax'] = min_max_scaler.fit_transform(riders_df[['age']])
print("\nData after Min-Max scaling:\n")
print(riders_df[['age', 'age_minmax']].head())


Data after Min-Max scaling:

   age  age_minmax
0   19    0.021277
1   33    0.319149
2   65    1.000000
3   65    1.000000
4   23    0.106383


C:\Users\victus\AppData\Local\Temp\ipykernel_614828\287486430.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  riders_df['age_minmax'] = min_max_scaler.fit_transform(riders_df[['age']])


## Q-6. Feature Construction

In [262]:
## avg_ride_distance
riders_df['avg_distance'] = trips_df['distance'].sum() / trips_df['distance'].count()
print("\nAverage Ride Distance:", riders_df['avg_distance'].iloc[0])

## avg_ride_fare
riders_df['avg_fare'] = trips_df['fare_amount'].sum() / trips_df['fare_amount'].count()
print("\nAverage Ride Fare:", riders_df['avg_fare'].iloc[0])

## is_peak_hour
trips_df['is_peak_hour'] = trips_df['hour'].apply(lambda x: 1 if (7 <= x <= 9) or (18 <= x <= 21) else 0)
print("\nPeak Hours Flag Created.")

## days_since_signup
trips_df = trips_df.merge(riders_df[['rider_id', 'signup_date']], on='rider_id', how='left')
trips_df['signup_date'] = pd.to_datetime(trips_df['signup_date'], errors='coerce')
trips_df['days_since_signup'] = (trips_df['pickup_datetime'] - trips_df['signup_date']).dt.days
print("\nDays Since Signup Feature Created.")   


## surge_flag
trips_df['surge_flag'] = trips_df['fare_amount'].apply(lambda x: 1 if x > avg_fare else 0)
print("\nSurge Flag Feature Created.")




Average Ride Distance: 1.594166394333558e-16

Average Ride Fare: -1.2753331154668464e-16

Peak Hours Flag Created.

Days Since Signup Feature Created.

Surge Flag Feature Created.


C:\Users\victus\AppData\Local\Temp\ipykernel_614828\1211364772.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  riders_df['avg_distance'] = trips_df['distance'].sum() / trips_df['distance'].count()
C:\Users\victus\AppData\Local\Temp\ipykernel_614828\1211364772.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  riders_df['avg_fare'] = trips_df['fare_amount'].sum() / trips_df['fare_amount'].count()


## Q-7. Final Dataset

In [264]:
rider_features = riders_df[['rider_id', 'age', 'gender', 'customer_ride_frequency', 'avg_distance', 'avg_fare']]
trip_features = trips_df[['trip_id', 'rider_id', 'fare_amount', 'distance', 'duration', 'hour', 'day_of_week', 'month', 'is_peak_hour', 'days_since_signup', 'surge_flag']]   
city_features = pd.concat([city_df[['zone_id', 'zone_name', 'traffic_level']], riders_df[['rider_id', 'customer_ride_frequency']]], axis=1)

print("\nRider Features:\n", rider_features.head())
print("\nTrip Features:\n", trip_features.head())
print("\nCity Zone Features:\n", city_features.head())



Rider Features:
   rider_id  age  gender customer_ride_frequency  avg_distance      avg_fare
0    R_001   19       2                     Low  1.594166e-16 -1.275333e-16
1    R_002   33       1                    High  1.594166e-16 -1.275333e-16
2    R_003   65       1                     Low  1.594166e-16 -1.275333e-16
3    R_004   65       2                    High  1.594166e-16 -1.275333e-16
4    R_005   23       0                    High  1.594166e-16 -1.275333e-16

Trip Features:
   trip_id rider_id  fare_amount  distance  duration  hour day_of_week  month  \
0  T_0000    R_004    -0.899310 -0.989000 -0.736929   2.0      Friday  April   
1  T_0001    R_032     1.435420  0.837341  1.232645  14.0    Saturday  April   
2  T_0002    R_011    -1.187877 -1.364885 -1.212266  14.0      Monday  April   
3  T_0003    R_003     0.645535 -0.171235 -0.371199   0.0      Friday  April   
4  T_0004    R_021    -0.784224 -0.739367 -0.969565   NaN         NaN    NaN   

   is_peak_hour  days_since_

In [265]:
## MAke a final csv file

final_df = pd.concat([rider_features, trip_features, city_features], axis=1)
final_df.to_csv('final_preprocessed_data.csv', index=False)
print("\nFinal preprocessed data saved to 'final_preprocessed_data.csv'.")


Final preprocessed data saved to 'final_preprocessed_data.csv'.
